# 3. Model Training: Tiny Transformer Multimodal Fusion
This notebook trains our Method B student: a tiny multi-layer Transformer Encoder that treats visual and audio representations as sequence tokens, prepends a `[CLS]` token, and projects the final state to the teacher embedding space.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import sys
sys.path.append('.')
from src.models import TransformerFusionApproximator
from src.loss import InfoNCELoss
from src.dataset import MultimodalEmbeddingDataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


## Step 1: Load datasets


In [ ]:
train_dataset = MultimodalEmbeddingDataset(file_path="train_features.pt")
test_dataset = MultimodalEmbeddingDataset(file_path="test_features.pt")

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)


## Step 2: Train Method B (Tiny Transformer Fusion)
We optimize the model using a joint loss: $L = L_{cos} + 0.5 \cdot L_{InfoNCE}$ to combine absolute alignment and relative cross-modal contrast.


In [ ]:
model_trans = TransformerFusionApproximator(img_dim=512, aud_dim=128, embed_dim=256, num_heads=4, num_layers=2, output_dim=1024).to(device)
optimizer = optim.AdamW(model_trans.parameters(), lr=1e-3, weight_decay=1e-4)

cosine_loss_fn = lambda pred, target: (1 - torch.nn.functional.cosine_similarity(pred, target)).mean()
infonce_loss_fn = InfoNCELoss(temperature=0.07, symmetric=True)

epochs = 30
for epoch in range(epochs):
    model_trans.train()
    epoch_loss = 0.0
    for batch in train_loader:
        z_img = batch['z_img'].to(device)
        z_aud = batch['z_aud'].to(device)
        v_teacher = batch['v_teacher'].to(device)
        
        optimizer.zero_grad()
        v_pred = model_trans(z_img, z_aud)
        
        # Joint loss
        loss_cos = cosine_loss_fn(v_pred, v_teacher)
        loss_nce = infonce_loss_fn(v_pred, v_teacher)
        loss = loss_cos + 0.5 * loss_nce
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item() * z_img.size(0)
        
    train_loss = epoch_loss / len(train_dataset)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:02d}/{epochs:02d} | Train Joint Loss: {train_loss:.4f}")

# Save the model weights
torch.save(model_trans.state_dict(), "transformer_fusion.pt")
print("Transformer training complete and weights saved!")
